In [ ]:
import os

import flatdict as fd
import requests
import yaml

print(os.getcwd())

In [ ]:
# url = "https://ndownloader.figshare.com/files/42742570"
# output = "test.pos"
def download_requests(url: str, output_file_path: str) -> bool:
    r = requests.get(url, stream=True, allow_redirects=True)
    if r.status_code == 200:
        # print(f"{r.url}, {r.status_code}, {r.headers}")
        with open(output_file_path, "wb") as fp:
            for chunk in r.iter_content(1024 * 1024):
                fp.write(chunk)
        # if "Content-Length" in r.headers:
        #     if int(os.path.getsize(output_file_path)) == int(r.headers["Content-Length"]):
        return True
        # else:
        #     return False
    else:
        return False


# status = download_requests(url, output)
# print(status)

In [ ]:
with open("data/datasets.yaml", encoding="utf-8") as fp:
    datasets = fd.FlatDict(yaml.safe_load(fp) or {}, delimiter="/")
for key, value in datasets.items():
    if key.endswith(r"\@origin"):
        case = key.rsplit("/", 1)[0]
        print(case)
        if os.path.isfile(f"data/{value}"):  # local file, never compressed by default
            print(f"local file, not compressed >>>> data/{value}")
        elif value.startswith("https://"):  # file to download
            if value.count(":") == 2:  # compressed
                archive_link, data_file_name = value.rsplit(":", 1)
                print(
                    f"remote file, compressed >>>> {archive_link}, {data_file_name}, {os.getcwd()}/{archive_link.rsplit('/')[-1]}"
                )
                status = download_requests(
                    archive_link, f"{archive_link.rsplit('/')[-1]}"
                )
            else:
                print(f"remote file, not compressed >>>> {value}")
                status = download_requests(value, f"{value.rsplit('/')[-1]}")
        else:
            continue